# Phase 2 — Transformer with learned Q / R–style noise

Kalman filters use **Q** (process noise) and **R** (measurement noise) to weight prediction vs observation. Here the motion head predicts, per time step and bbox dimension:

- **log_var_q** — heteroscedastic variance on **innovation / residual** (learned process uncertainty; analogous to how much the state may drift).
- **log_var_r** — optional **measurement** uncertainty; training can align it with squared error between noisy inputs and clean GT on the history window (detector noise proxy).

Training minimizes a **Gaussian negative log-likelihood** on bbox errors (with variance from `softplus(log_var_q)`), plus **CIoU** and **confidence** terms (`LearnedNoiseMotionLoss` in `learned_noise_motion.py`). At inference, high variance steps can be down-weighted when fusing with detections or gating associations.

Adjust `BASE_DIR` to your MOT-style dataset roots (same layout as `phase1_improved.ipynb`).

In [4]:
import os
import torch
from torch import optim
from torch.utils.data import DataLoader

from dataset import GTSequenceDataset
from learned_noise_motion import MotionTransformerLearnedNoise, LearnedNoiseMotionLoss

SEQ_IN_LEN = 30
SEQ_OUT_LEN = 20
SEQ_TOTAL_LEN = 50
BATCH_SIZE = 512
STEPS = 4
NOISE_COEFFICIENT = 0.15
NOISE_PROB = 0.2
BASE_DIR = os.environ.get("MOT_DATASET_ROOT", "../../.Datasets/")

train_dataset = GTSequenceDataset.from_roots(
    [
        # f'{BASE_DIR}/SportsMOT/train',
        # f'{BASE_DIR}DanceTrack/train',
        f'{BASE_DIR}MOT17/train',
        # f'{BASE_DIR}MOT20/train'
    ],
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    noise_coeff=NOISE_COEFFICIENT,
    noise_prob=NOISE_PROB,
)
val_dataset = GTSequenceDataset.from_roots(
    [
        # f'{BASE_DIR}/SportsMOT/val',
        # f'{BASE_DIR}DanceTrack/val',
        f'{BASE_DIR}MOT17/val',
        # f'{BASE_DIR}MOT20/val'
    ],
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    noise_coeff=NOISE_COEFFICIENT,
    noise_prob=NOISE_PROB,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 15072, Val: 28163


In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 3e-4
NUM_EPOCHS = 15

model = MotionTransformerLearnedNoise(
    input_dim=13,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_ff=1024,
    dropout=0.1,
).to(DEVICE)

criterion = LearnedNoiseMotionLoss(
    nll_coeff=1.0,
    ciou_coeff=0.5,
    conf_coeff=0.25,
    r_supervise_coeff=0.1,
)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(NUM_EPOCHS, 1))

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,} | device={DEVICE}")

Parameters: 3,244,557 | device=cuda


In [6]:
best = float("inf")
os.makedirs("pretrained", exist_ok=True)
ckpt = "pretrained/transformer_learned_qr.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    tr = model.train_one_epoch(train_loader, optimizer, criterion, device=DEVICE)
    va = model.evaluate(val_loader, criterion, device=DEVICE)
    scheduler.step()
    if va < best:
        best = va
        model.save_weight(ckpt)
    print(f"epoch {epoch:03d}  train {tr:.5f}  val {va:.5f}  lr {scheduler.get_last_lr()[0]:.2e}")

print("best val loss:", best, "saved:", ckpt)

KeyboardInterrupt: 

In [ ]:
# Quick rollout sanity check (first batch)
from learned_noise_motion import softplus_var

model.eval()
src, trg, gt_src, gt_trg = next(iter(val_loader))
src = src.to(DEVICE)
trg = trg.to(DEVICE)
with torch.no_grad():
    pred, log_vq, log_vr = model(src, trg[:, :-1])
    vq = softplus_var(log_vq).mean().item()
    vr = softplus_var(log_vr).mean().item()
print("mean softplus(var_q):", round(vq, 6), "mean softplus(var_r):", round(vr, 6))
print("pred shape:", tuple(pred.shape))